# SQL Agent · LangGraph 进阶版

上一版（`07-sql-agent-langchain`）已经用 LangGraph `StateGraph` 把 7 阶段流水线**画成了一张图**，
但那张图是**无状态、跑完即散**的。本版在**不改动任何既有行为**的前提下，补上 LangGraph
真正有别于「手写 for/if」的三项能力 —— 而且每个缺口，都是上一版 README 自己写下的。

## 这个 notebook 演示什么

| 节 | 内容 | 对应的 LangGraph 机制 |
|---|---|---|
| 1 | 会话图拓扑（父图 + SQL 子图） | `StateGraph` · 子图 `add_node("sql", subgraph)` |
| 2 | **持久化**：多轮记忆落检查点 | `checkpointer` · `thread_id` · `get_state` |
| 3 | **澄清续接**：反问→暂停→续接 | `interrupt()` · `Command(resume=...)` |
| 4 | **人工介入**：执行前复核 SQL | `interrupt()` 的 approve / edit / reject |
| 5 | **时间旅行**：轨迹与回放 | `get_state_history()` · `update_state()` |
| 6 | 命令行入口 | `--thread-id` / `--review` / `--trajectory` / `--replay` |

> 一句话：**上一版证明「换框架 ≠ 换行为」；本版证明「换框架 ≠ 白换」。**
> 设计取舍见 [DESIGN.md](DESIGN.md)。

## 0. 环境准备（脱敏加载 `.env` + 连接数据库）

> 每个代码格都**自包含**（顶部按需重建 `parser`）：重启内核后只点单格的「运行」，
> 也不会再出现 `NameError: name 'parser' is not defined`。


In [21]:
from sqlagent.config import get_db_url, get_llm_settings, load_env, redact
from sqlagent.db import DBParser

env_path = load_env()
db_url = get_db_url()

print("已加载 .env :", redact(env_path))
print("数据库       :", redact(db_url))
print("模型设置     :", get_llm_settings().describe())   # 脱敏：不含 Key

parser = DBParser(db_url, verbose=False)
print("表数量       :", len(parser.table_names))

已加载 .env : ~/Downloads/padow-ai/.env
数据库       : sqlite:///~/Downloads/padow-ai/Week13/Week13/sql-agent-LangGraph-advanced/chinook.db
模型设置     : provider=deepseek model=deepseek-v4-flash base_url=https://api.deepseek.com/v1
表数量       : 11


## 1. 会话图：把「一整轮对话」也画成一张图

上一版里，「三道闸门 + 多轮对话」是 `ChatBI` 里的**纯 Python 编排**（`if/elif`），
只有进到 `in` 分支才交给 LangGraph。本版把这层也纳入图 —— 因为**要暂停、要持久化，
状态就必须先在图里**。

```mermaid
flowchart LR
    START --> rewrite --> scope
    scope -- in --> sql --> finalize --> END
    scope -- out --> reject --> finalize
    scope -- ambiguous --> clarify --> rewrite
```

- `rewrite` ← [A] 追问改写（并重置本轮字段，防止上一轮串味）；
- `scope`   ← [B] 范围判定（声称 `in` 却没落到真实表 → 结构性降级为 `ambiguous`）；
- `sql`     ← **SQL 子图**（上一版的 7 阶段图），以 `add_node("sql", subgraph)` 挂载；
- `clarify` ← `interrupt()` 暂停，等用户回答后续接回 `rewrite`；
- `reject`  ← 越界问题直接谢绝，**不生成任何 SQL**；
- `finalize`← 把本轮 `(问题, 回答)` 追加进 `turns`，供下一轮追问改写使用。

> 与「把 7 阶段摊平进一张大图」相比：**子图让 SQL 流水线保持独立可跑**（旧测试原样通过），
> 父图只负责「对话层」的编排。

In [16]:
from sqlagent.config import get_db_url
from sqlagent.db import DBParser

parser = DBParser(get_db_url(), verbose=False)   # 自包含：本格可脱离其它代码格单独运行

from sqlagent.conversation import ConversationBI

bot = ConversationBI(parser, thread_id="demo-topo", verbose=False)
print(bot.graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	rewrite(rewrite)
	scope(scope)
	clarify(clarify)
	sql(sql)
	reject(reject)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> rewrite;
	clarify --> rewrite;
	reject --> finalize;
	rewrite --> scope;
	scope -.-> clarify;
	scope -.-> reject;
	scope -.-> sql;
	sql --> finalize;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [1]:
# 自证：notebook 里看不到 langgraph 的 import，是因为它藏在 sqlagent 包内部 ——
# 但运行时的「对象身份」骗不了人：这张图就是 langgraph 的 CompiledStateGraph。
import sys
from importlib.metadata import version

import sqlagent.conversation.graph as cg
from sqlagent.config import get_db_url
from sqlagent.conversation import ConversationBI
from sqlagent.db import DBParser

proof = ConversationBI(DBParser(get_db_url(), verbose=False),
                       thread_id="demo-proof", verbose=False)   # 仅构图，不调用模型

print("langgraph 版本              :", version("langgraph"))
print("构图用的 StateGraph         :", f"{cg.StateGraph.__module__}.{cg.StateGraph.__name__}")
print("interrupt 来源              :", cg.interrupt.__module__)
print("proof.graph 实际类型        :", f"{type(proof.graph).__module__}.{type(proof.graph).__name__}")
print("  继承链                    :", " -> ".join(c.__name__ for c in type(proof.graph).__mro__[:4]))
print("图上的节点                  :", sorted(proof.graph.get_graph().nodes))
print("checkpointer 类型（父图持有）:", type(proof.checkpointer).__name__)
print("已加载 langgraph.* 子模块数 :", sum(m.startswith("langgraph") for m in sys.modules))


langgraph 版本              : 1.2.10
构图用的 StateGraph         : langgraph.graph.state.StateGraph
interrupt 来源              : langgraph.types
proof.graph 实际类型        : langgraph.graph.state.CompiledStateGraph
  继承链                    : CompiledStateGraph -> Pregel -> PregelProtocol -> Runnable
图上的节点                  : ['__end__', '__start__', 'clarify', 'finalize', 'reject', 'rewrite', 'scope', 'sql']
checkpointer 类型（父图持有）: InMemorySaver
已加载 langgraph.* 子模块数 : 127


## 2. 持久化：Checkpointer + `thread_id`

**问题**：上一版的多轮记忆是 `ChatBI` 里的一个 Python `list` —— 进程一退就没了。

**做法**：给图接上 checkpointer，所有中间状态按 `thread_id` 落库。
`make_checkpointer(kind)` 提供两档：`memory`（`InMemorySaver`，默认）与
`sqlite`（`SqliteSaver`，落盘，**跨进程重启也能恢复**）。

先看内存版：多轮对话的 `turns` 不再放 Python 变量，而是**读自检查点**。

In [23]:
from sqlagent.config import get_db_url
from sqlagent.conversation import ConversationBI
from sqlagent.db import DBParser

parser = DBParser(get_db_url(), verbose=False)   # 自包含：本格可脱离其它代码格单独运行

bot = ConversationBI(parser, thread_id="demo-memory")
question_list = [
    # "数据库里一共有多少首歌曲？", 
    # "销量最高的 5 个流派分别是什么？",
    "销量最高的前5名歌手是谁？各自的销量是多少？"
]
for q in question_list:
    trace = bot.ask(q)
    print(f"用户：{q}")
    print(f"SQL ：{trace['sql']}")          # 把实际执行的 SQL 一并返回，方便核对统计口径
    print(f"助手：{trace['answer']}")
    print("-" * 60)

print("会话历史（读自检查点）：")
for q, a in bot.turns:
    print(f"  - {q}  →  {a[:40]}...")


用户：销量最高的前5名歌手是谁？各自的销量是多少？
SQL ：SELECT ar.Name, SUM(ii.Quantity) AS Sales
FROM invoice_items ii
JOIN tracks t ON ii.TrackId = t.TrackId
JOIN albums al ON t.AlbumId = al.AlbumId
JOIN artists ar ON al.ArtistId = ar.ArtistId
GROUP BY ar.ArtistId, ar.Name
ORDER BY Sales DESC
LIMIT 5;
助手：销量最高的前5名歌手及销量如下：

1. Iron Maiden：140  
2. U2：107  
3. Metallica：91  
4. Led Zeppelin：87  
5. Os Paralamas Do Sucesso：45
------------------------------------------------------------
会话历史（读自检查点）：
  - 销量最高的前5名歌手是谁？各自的销量是多少？  →  销量最高的前5名歌手及销量如下：

1. Iron Maiden：140  
2...


再看 sqlite 版：换一个**全新的 `ConversationBI` 实例**（模拟进程重启），
只要 `thread_id` 与落盘文件不变，就能读到上一次的会话历史。

In [20]:
from sqlagent.config import get_db_url
from sqlagent.conversation import ConversationBI
from sqlagent.db import DBParser

parser = DBParser(get_db_url(), verbose=False)   # 自包含：本格可脱离其它代码格单独运行

CKPT = "checkpoints-demo.sqlite"

first = ConversationBI(parser, thread_id="demo-sqlite", checkpointer_kind="sqlite",
                       checkpointer_path=CKPT)
first.ask("数据库里一共多少为歌手？每位歌手有多少张专辑？")

# 模拟「进程重启」：新建实例、重建检查点连接，仅凭 thread_id 找回历史
second = ConversationBI(parser, thread_id="demo-sqlite", checkpointer_kind="sqlite",
                        checkpointer_path=CKPT)
print("重启后仍能读到的会话历史：", second.turns)

重启后仍能读到的会话历史： [('数据库里一共多少为歌手？每位歌手有多少张专辑？', '数据库中共有 **275 位歌手**。\n\n查询结果按每位歌手统计了专辑数量：`album_count` 表示该歌手的专辑数，包含 **0 张专辑**的歌手。例如：Led Zeppelin 有 14 张，Metallica 有 10 张，Various Artists 有 4 张，Audioslave 有 3 张，AC/DC 有 2 张；也有部分歌手如 Milton Nascimento & Bebeto、Azymuth、João Gilberto 等为 0 张。\n\n由于结果共 275 行、当前仅展示前 50 行，无法在此完整列出全部 275 位歌手各自的专辑数；完整明细需查看全部查询结果。')]


## 3. 澄清续接：反问 → 暂停 → 续接

**问题**：上一版 README 自述「澄清反问为**单轮**」——`ambiguous` 分支给出澄清语后，
只能等用户下一条消息从头发起，**没有「反问→等待→续接」的状态机**。

**做法**：`ambiguous` 分支改为 `clarify` 节点，内部 `interrupt()` **暂停**并把反问抛到图外；
用户回答通过 `Command(resume=...)` 送回，图从暂停处**续接**回 `rewrite`。

In [5]:
from sqlagent.config import get_db_url
from sqlagent.conversation import ConversationBI
from sqlagent.db import DBParser

parser = DBParser(get_db_url(), verbose=False)   # 自包含：本格可脱离其它代码格单独运行

bot = ConversationBI(parser, thread_id="demo-clarify")

trace = bot.ask("有多少？")          # 意图不完整 → 图在此暂停
print("状态：", trace["status"])
print("反问：", trace["interrupt"]["question"])

trace = bot.resume("数据库里一共有多少首歌曲？")   # 用户澄清 → 续接
print("续接后状态：", trace["status"])
print("助手：", trace["answer"])

状态： interrupted
反问： 您想统计哪个对象的数量？例如：曲目数、专辑数、歌手数、客户数、发票数等。


续接后状态： done
助手： 数据库里一共有 **3503 首歌曲**。


In [24]:
# 【交互式格】在 IDE / Jupyter 里直接运行即可手动输入问题（多轮 + 追问）。
# 已打 skip-execution 标记：无头执行（nbconvert --execute）会跳过本格，避免 input() 阻塞。
from sqlagent.config import get_db_url
from sqlagent.conversation import ConversationBI
from sqlagent.db import DBParser

bot = ConversationBI(DBParser(get_db_url(), verbose=False), thread_id="my-chat")

while True:
    q = input("你：").strip()
    if q in ("", "exit"):
        break
    trace = bot.ask(q)
    # 遇到澄清 / 复核暂停 → 就地回答，再把答复送回图
    while trace["status"] == "interrupted":
        payload = trace["interrupt"]
        print("助手：" + ("\n".join(payload.get("question", "")) if isinstance(payload.get("question"), list) else payload.get("question", "")))
        trace = bot.resume(input("你：").strip())
    print("助手：", trace["answer"])

print("本会话历史：", bot.turns)      # 读自检查点

助手： 共有 25 个流派。
助手： 销量最高的3个流派分别是：

1. **Rock**：835  
2. **Latin**：386  
3. **Metal**：264
助手： 在 Metal 流派下，共有 **14 位歌手**。按该 SQL 统计的销量如下：

- Apocalyptica：4
- Black Label Society：8
- Black Sabbath：9
- Bruce Dickinson：12
- Godsmack：8
- Guns N' Roses：10
- Iron Maiden：70
- Judas Priest：10
- Lenny Kravitz：7
- Metallica：91
- Motörhead：6
- Mötley Crüe：7
- Ozzy Osbourne：10
- System Of A Down：12

以上共 14 位歌手，销量合计为 **264**。
本会话历史： [('有多少个流派？', '共有 25 个流派。'), ('销量最高的3个流派是哪3个？销量分别是多少？', '销量最高的3个流派分别是：\n\n1. **Rock**：835  \n2. **Latin**：386  \n3. **Metal**：264'), ('Metal流派下，共有多少歌手？每个歌手销售了多少张专辑？', "在 Metal 流派下，共有 **14 位歌手**。按该 SQL 统计的销量如下：\n\n- Apocalyptica：4\n- Black Label Society：8\n- Black Sabbath：9\n- Bruce Dickinson：12\n- Godsmack：8\n- Guns N' Roses：10\n- Iron Maiden：70\n- Judas Priest：10\n- Lenny Kravitz：7\n- Metallica：91\n- Motörhead：6\n- Mötley Crüe：7\n- Ozzy Osbourne：10\n- System Of A Down：12\n\n以上共 14 位歌手，销量合计为 **264**。")]


## 4. 人工介入：执行前的复核闸门

**问题**：模型生成 SQL 后**直接执行**，中间没有任何人把关。

**做法**：`human_review=True` 时，在 `check` 通过之后、`execute` 之前插入 `human_review` 节点，
`interrupt()` 把待审 SQL 抛出来，等外部三种决定之一：

```
check(通过) → human_review ─┬─ approve → execute
                            ├─ edit    → check（人工改写的 SQL 必须重新过守卫与试跑）
                            └─ reject  → fail
```

In [35]:
from sqlagent.config import get_db_url
from sqlagent.conversation import ConversationBI
from sqlagent.db import DBParser

parser = DBParser(get_db_url(), verbose=False)   # 自包含：本格可脱离其它代码格单独运行

bot = ConversationBI(parser, thread_id="demo-hitl-approve", human_review=True)

question_list = [
    # "数据库里一共有多少首歌曲？", 
    # "销量最高的 5 个流派分别是什么？",
    "歌曲最多5名歌手是谁？各自名下有多少首歌曲？",
    "专辑最多5名歌手是谁？各自名下有多少专辑？",
    "今天天气如何？"          # ← out 分支：不进 SQL 子图，直接谢绝
]

trace = bot.ask(question_list[-1])
print("状态：", trace["status"])

# 非 in 的中断有两种、载荷不同：sql_review 是复核闸门，clarify 是澄清反问；
# 而 out 的问题根本不会中断（在进 SQL 子图之前就谢绝了），所以必须先判 status。
if trace["status"] == "interrupted":
    kind = trace["interrupt"]["type"]
    print("暂停类型：", kind)
    if kind == "sql_review":
        print("待审 SQL：", trace["interrupt"]["sql"])
        trace = bot.resume({"action": "approve"})      # 放行
    elif kind == "clarify":                             # 澄清反问：续接值是字符串
        trace = bot.resume("数据库里一共有多少首歌曲？")
    print("结果：", trace["rows"])
else:
    print("范围：", trace["scope"]["scope"], "·", trace["scope"]["reason"])
print("助手：", trace["answer"])

状态： done
范围： out · 天气信息与 chinook 音乐商店数据库无关，且无法通过所列表和字段查询。
助手： 抱歉，这个问题超出了当前 chinook 数据库的范围：天气信息与 chinook 音乐商店数据库无关，且无法通过所列表和字段查询。
你可以问我销量 / 客户 / 员工 / 曲目 / 专辑 / 流派等数据问题。


In [27]:
from sqlagent.config import get_db_url
from sqlagent.conversation import ConversationBI
from sqlagent.db import DBParser

parser = DBParser(get_db_url(), verbose=False)   # 自包含：本格可脱离其它代码格单独运行

# 驳回：直接走 fail，绝不执行、绝不编造
bot = ConversationBI(parser, thread_id="demo-hitl-reject", human_review=True)
trace = bot.ask(question_list[0])
trace = bot.resume({"action": "reject", "note": "这个口径不对"})
print("状态：", trace["status"], "| ok =", trace["ok"], "| rows =", trace["rows"])
print("助手：", trace["answer"])

状态： done | ok = False | rows = None
助手： 抱歉，未能生成可执行的 SQL，已放弃（不编造结果）。


In [28]:
from sqlagent.config import get_db_url
from sqlagent.conversation import ConversationBI
from sqlagent.db import DBParser

parser = DBParser(get_db_url(), verbose=False)   # 自包含：本格可脱离其它代码格单独运行

# 人工改写：把 SQL 换成人工认可的版本 —— 仍要重新过守卫与试跑，但不再要求本人二次审批
bot = ConversationBI(parser, thread_id="demo-hitl-edit", human_review=True)
bot.ask("数据库里一共有多少首歌曲？")
trace = bot.resume({"action": "edit", "sql": "SELECT COUNT(*) FROM albums;"})
print("改写后 SQL：", trace["sql"])
print("改写后结果：", trace["rows"])
print("助手：", trace["answer"])

改写后 SQL： SELECT COUNT(*) FROM albums;
改写后结果： [(347,)]
助手： 根据执行结果，`albums` 表中共有 **347 条记录**。

需要说明的是：本次 SQL 统计的是 `albums`（专辑）表的行数，而非歌曲表。因此 347 是专辑表的记录数，不能直接等同于歌曲数量。如需准确统计歌曲总数，应改为对歌曲表执行 `SELECT COUNT(*)`。


## 5. 时间旅行：轨迹与回放

**问题**：上一版失败定位靠**手写 `trace` 字典**，想在「同一处境」下换写法做 A/B 复盘，无迹可寻。

**做法**：接上 checkpointer 后，`graph.get_state_history(config)` 原生吐出一整条**检查点链**
（每一跳的状态快照）。`list_hops()` 把它转成可读轨迹，`replay_from()` 则从**任意一跳**继续跑。

> 时间旅行对**任何**接了 checkpointer 的图都成立。这里为了演示「从 `execute` 前那一跳回放」，
> 用单图形态（`build_checkpointed_graph`）——因为会话图里 `execute` 藏在子图内部，
> 父图看到的「下一跳」是 `sql`。

In [ ]:
from sqlagent.config import get_db_url
from sqlagent.db import DBParser
from sqlagent.session import make_checkpointer, thread_config
from sqlagent.time_travel import checkpoint_before, format_hops, list_hops, replay_from
from sqlagent.workflow import build_checkpointed_graph

parser = DBParser(get_db_url(), verbose=False)   # 自包含：本格可脱离其它代码格单独运行

graph = build_checkpointed_graph(parser, checkpointer=make_checkpointer(), verbose=False)
cfg = thread_config("demo-travel")
initial = {"question": "数据库里一共有多少首歌曲？", "attempts": 0, "ok": False,
           "error": None, "columns": None, "rows": None, "answer": None}

first = graph.invoke(initial, cfg)
print("首次执行 rows：", first["rows"])
print()
print(format_hops(list_hops(graph, cfg)))

# ── A/B 回放：定位「下一跳将进入 execute」的检查点，换一条 SQL 从那一跳分叉重跑 ──
cid = checkpoint_before(graph, cfg, "execute")
print("定位到检查点：", cid)

final = replay_from(graph, cfg, cid, updates={"sql": "SELECT COUNT(*) FROM albums;"})
print("首次（歌曲数）rows ：", first["rows"])
print("回放（专辑数）rows ：", final["rows"])
print("回放后 answer      ：", final["answer"])


首次执行 rows： [(3503,)]

共 7 跳（最新在前）：
[00] step=5 下一跳=END attempts=1 ok=True
     SQL: SELECT COUNT(*) AS song_count FROM tracks;
[01] step=4 下一跳=summarize attempts=1 ok=True
     SQL: SELECT COUNT(*) AS song_count FROM tracks;
[02] step=3 下一跳=execute attempts=1 ok=True
     SQL: SELECT COUNT(*) AS song_count FROM tracks;
[03] step=2 下一跳=check attempts=0 ok=False
     SQL: SELECT COUNT(*) AS song_count FROM tracks;
[04] step=1 下一跳=generate attempts=0 ok=False
     SQL: None
[05] step=0 下一跳=select attempts=0 ok=False
     SQL: None
[06] step=-1 下一跳=__start__ attempts=None ok=None
     SQL: None
定位到检查点： 1f1e5b79-085b-63b4-8003-c18a8e662ad6
首次（歌曲数）rows ： [(3503,)]
回放（专辑数）rows ： [(347,)]
回放后 answer      ： 根据执行的 SQL 结果，查询返回的计数为 **347**。

不过需要说明：该 SQL 统计的是 `albums`（专辑）表的记录数，严格来说这是专辑数量，并不等同于歌曲数量。若需歌曲总数，应对歌曲表（如 `songs`/`tracks`）执行 `COUNT(*)` 才能得到准确答案。


## 6. 命令行入口（脱离 notebook 也能跑）

```bash
# 基线（与上一版一致）
python -m sqlagent.cli "数据库里一共有多少首歌曲？"

# 会话图 + 持久化：指定 thread-id，多轮记忆落库
python -m sqlagent.cli --thread-id my-chat --repl

# 人工复核：生成 SQL 后暂停，等 approve / edit / reject
python -m sqlagent.cli "有多少首歌？" --review

# 时间旅行：跑完打印检查点轨迹 / 从某一跳回放
python -m sqlagent.cli "有多少首歌？" --thread-id my-chat --trajectory
python -m sqlagent.cli --thread-id my-chat --replay <checkpoint_id>

# sqlite 落盘（跨进程恢复）
python -m sqlagent.cli --thread-id my-chat --checkpointer sqlite --checkpoint-path ck.sqlite
```

离线单测（68 项，零网络，不需要 Key）：

```bash
pytest
```

## 7. 小结

- **三件套一一对应上一版 README 自陈的缺口**：多轮记忆 / 澄清单轮 → 持久化；
  生成 SQL 无人在环 → 人工介入；手写 trace → 时间旅行。
- **行为不变量始终守住**：`run_workflow(question, parser)` 签名与返回字段逐字不变；
  默认路径（`human_review=False`）LLM 调用次数不变；既有单测原样通过（增量新增，不破坏）。
- **架构选择**：会话图 = 父图（对话层）+ SQL 子图（流水线层）。
  子图不带 checkpointer（LangGraph 硬性要求），checkpointer 由父图统一持有 —— 于是
  子图仍可单独跑，父图负责暂停与记忆。
- 详见 [DESIGN.md](DESIGN.md) 的「为什么长这样」。